In [77]:
# Test DataFrame with text data
data = {
    'text': [
        'I need some help with shopping',
        'Just checking in to say hello.',
        'Can someone help me with this issue?',
        'Looking forward to the weekend!',
        'Just checking in to say hello.'
    ]*1000 
}
df = pd.DataFrame(data)

## Automating Closed Coding using lanauge models 

In [78]:
import pandas as pd
from pydantic import BaseModel, ValidationError
import json
from ollama import chat

# Define the Pydantic model for structured response
class HelpRequest(BaseModel):
    help: bool


# Function to annotate each text entry
def annotate_help_request(text):
    prompt = (
        f'You are an expert annotator in the social sciences and you have to determine if the following message is a request for help, either indirectly or directly.: "{text}" '
        'Respond with a JSON object: {"help": true} or {"help": false}.'
    )
    try:
        response = chat(
            model='llama3.2',  # Use a more capable model for better accuracy
            messages=[{'role': 'user', 'content': prompt}],
            format=HelpRequest.model_json_schema(),
        )
        response_content = response.get('message', {}).get('content', '')
        help_data = json.loads(response_content)
        help_request = HelpRequest(**help_data)
        return help_request.help
    except (json.JSONDecodeError, ValidationError, KeyError) as e:
        print(f"Error processing text: {text}\nError: {e}")
        return None

In [79]:
# Apply the annotation function to the DataFrame
df['is_help_request'] = df['text'].apply(annotate_help_request)

In [80]:
# Display the annotated DataFrame
df['is_help_request'].value_counts()

is_help_request
False    3051
True     1949
Name: count, dtype: int64

# Description of the Code

## 🧠 Function Overview

The `annotate_help_request` function determines whether a given text is a request for help by leveraging a language model through the Ollama Python library and validating the response using a Pydantic model.

* **Input**: A string `text` representing the message to be analyzed.
* **Output**: A boolean value indicating the classification result (`True` if it's a help request, `False` otherwise).

---

## 🧱 Step-by-Step Breakdown

### 1. **Prompt Construction**

```python
prompt = (
    f'You are an expert annotator in the social sciences and you have to determine if the following message is a request for help, either indirectly or directly.: "{text}" '
    'Respond with a JSON object: {"help": true} or {"help": false}.'
)
```



* **Purpose**: Creates a prompt string that instructs the language model to analyze the input `text` and determine if it's a help request.
* **Details**:

  * Sets the context by assigning the model the role of an "expert annotator in the social sciences."
  * Includes the input `text` within quotation marks for clarity.
  * Explicitly instructs the model to respond with a JSON object indicating the result.

### 2. **Model Invocation**

```python
response = chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': prompt}],
    format=HelpRequest.model_json_schema(),
)
```



* **Purpose**: Sends the constructed prompt to the specified language model and requests a structured response.
* **Details**:

  * `model='llama3.2'`: Specifies the use of the `llama3.2` model.
  * `messages`: Provides the prompt to the model as a user message.
  * `format=HelpRequest.model_json_schema()`: Supplies the expected JSON schema (defined by the `HelpRequest` Pydantic model) to guide the model's response format.

### 3. **Response Extraction**

```python
response_content = response.get('message', {}).get('content', '')
```



* **Purpose**: Extracts the content of the model's response from the nested dictionary structure.
* **Details**:

  * `response.get('message', {})`: Retrieves the 'message' dictionary from the response; defaults to an empty dictionary if 'message' is not present.
  * `.get('content', '')`: Retrieves the 'content' string from the 'message' dictionary; defaults to an empty string if 'content' is not present.

### 4. **JSON Parsing**

```python
help_data = json.loads(response_content)
```



* **Purpose**: Parses the JSON-formatted string `response_content` into a Python dictionary.
* **Details**:

  * `json.loads()`: Converts a JSON string into a Python dictionary.
  * Assumes that `response_content` is a valid JSON string matching the expected schema.

### 5. **Pydantic Model Validation**

```python
help_request = HelpRequest(**help_data)
```



* **Purpose**: Validates and structures the parsed JSON data using the `HelpRequest` Pydantic model.
* **Details**:

  * `HelpRequest` is a Pydantic model defined elsewhere in the code, likely as:

    ```python
    from pydantic import BaseModel

    class HelpRequest(BaseModel):
        help: bool
    ```

  * `HelpRequest(**help_data)`: Instantiates the model with the parsed data, ensuring it conforms to the expected schema (i.e., contains a boolean field 'help').

### 6. **Result Return**

```python
return help_request.help
```



* **Purpose**: Returns the value of the 'help' field from the validated `HelpRequest` model.
* **Details**:

  * This boolean value indicates whether the input text was classified as a help request (`True`) or not (`False`).

### 7. **Error Handling**

```python
except (json.JSONDecodeError, ValidationError, KeyError) as e:
    print(f"Error processing text: {text}\nError: {e}")
    return None
```



* **Purpose**: Catches and handles exceptions that may occur during JSON parsing or model validation.
* **Details**:

  * `json.JSONDecodeError`: Raised if `response_content` is not valid JSON.
  * `ValidationError`: Raised if the parsed data doesn't conform to the `HelpRequest` schema.
  * `KeyError`: Raised if expected keys are missing in the response dictionary.
  * Logs the error and returns `None` to indicate that the classification could not be performed.

---

## 📝 Summary

The `annotate_help_request` function automates the classification of text messages as help requests by:

1. Constructing a detailed prompt for the language model.
2. Sending the prompt to the model and requesting a structured JSON response.
3. Parsing and validating the model's response using a Pydantic schema.
4. Returning the classification result as a boolean value.
5. Handling any errors that may arise during the process.

This approach leverages the capabilities of language models to interpret nuanced text and provides structured outputs that can be easily integrated into data processing pipelines.

---

If you need further assistance or have additional questions, feel free to ask!


## Automating open and focused coding using lanuage models

In [92]:
import pandas as pd
from pydantic import BaseModel, ValidationError
import json
from ollama import chat
from pydantic import BaseModel, conlist, constr

class EmotionResponse(BaseModel):
     emotions: conlist(str, min_length=1, max_length=3)


def annotate_emotions(text):
    prompt = (
        f'You are an expert annotator in the social sciences engaged in the process of open coding reviews with a focus on the emotions expressed. '
        f'Provide one or more codes for this text: "{text}" '
        'Respond with a JSON object: {"emotions": [str]}'
    )
    try:
        response = chat(
            model='llama3.2',  # Use a more capable model for better accuracy
            messages=[{'role': 'user', 'content': prompt}],
            format=EmotionResponse.model_json_schema(),
        )
        response_content = response.get('message', {}).get('content', '')
        emotion_data = json.loads(response_content)
        emotion_response = EmotionResponse(**emotion_data)
        return emotion_response.emotions
    except (json.JSONDecodeError, ValidationError, KeyError) as e:
        print(f"Error processing text: {text}\nError: {e}")
        return None


In [93]:
text_input = "I feel overwhelmed and anxious about the upcoming exams."
emotions = annotate_emotions(text_input)
print(emotions)

['overwhelmed', 'anxious']
